In [1]:
!pip install transformers

In [2]:
from huggingface_hub import login

# IMPORTANT: HuggingFace Authentication Token Required
# This token allows access to gated models that require user agreement
# To get a token: Visit huggingface.co -> Settings -> Access Tokens -> Create new token
# Replace with your personal HuggingFace token for model access

from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Use Gemma 1B instruction-tuned (or swap for another small 1B LLM)
model_name = "google/gemma-3-1b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [4]:
def get_llm_reply(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=64)
    reply = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Strip prompt from output for cleaner reply
    return reply[len(prompt):].strip()

# print("Simple LLM chatbot (type 'quit' to exit)")
# while True:
#     text = input("You: ")
#     if text.lower().strip() == "quit":
#         print("Bot: Bye!")
#         break
#     print("Bot:", get_llm_reply(text))


In [5]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model_name,
    # use_fast=False       # Use slower but more reliable tokenizer
)
system_prompt = "You are a helpful assistant. Please reply clearly and briefly."
# system_prompt = "You are a helpful assistant. Please reply clearly."

messages = [{"role": "system", "content": system_prompt}]   # conversation history

print("Simple LLM chatbot (type 'quit' to exit)")
while True:
    text = input("You: ")
    if text.lower().strip() == "quit":
        print("Bot: Bye!")
        break
    messages.append({"role": "user", "content": text})
    response = pipe(messages, max_new_tokens=256)
    bot_reply = response[0]["generated_text"][-1]["content"].strip()   # last message = the new reply
    messages.append({"role": "assistant", "content": bot_reply})
    print("Bot:", bot_reply)

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Simple LLM chatbot (type 'quit' to exit)
You: how much do you know about Brunel University?
Bot: Brunel University is a highly regarded UK university with a strong reputation for engineering, business, and sciences. Here’s a breakdown of what I know:

*   **History:** Founded in 1876, it’s one of the oldest universities in the UK.
*   **Programs:** Known for its strong engineering programs (particularly in aerospace, mechanical, and electrical engineering), but also offers a range of other disciplines.
*   **Location:** Situated in Bath, Somerset, close to Stonehenge.
*   **Reputation:** Consistently ranked highly in UK university rankings.
*   **Notable Features:** It has a strong focus on innovation and research, particularly in areas like sustainable technologies.

---

Do you want me to tell you more about a specific aspect of Brunel University (e.g., a particular program, its history, or its ranking)?
You: do they have a computer science department?
Bot: Yes, Brunel University has